In [ ]:
from openbabel import pybel
from openbabel import openbabel as ob

def extract_ligands_split(xyz_file):
    mols = list(pybel.readfile("xyz", xyz_file))
    if not mols:
        print("Пустой файл")
        return
    
    mol = mols[0]
    
    # Список металлов (с Ir=77, Ru=44)
    metals_atomic_nums = {26, 28, 29, 30, 44, 46, 77, 78, 27, 25, 24, 42}
    
    # Удаляем металлы в обратном порядке индексов
    metal_atoms = [atom for atom in mol.atoms if atom.atomicnum in metals_atomic_nums]
    metal_atoms_sorted = sorted(metal_atoms, key=lambda a: a.idx, reverse=True)
    
    for atom in metal_atoms_sorted:
        mol.OBMol.DeleteAtom(atom.OBAtom)
    
    # **СПЛИТТИНГ по точке разрыва** - разбиваем на связные фрагменты
    fragments = mol.OBMol.Separate()
    
    ligands = []
    for i, frag_obmol in enumerate(fragments):
        ligand_mol = pybel.Molecule(frag_obmol)
        smiles = ligand_mol.write("smi").strip()
        
        ligand_xyz = f"ligand_{i+1}.xyz"
        ligand_mol.write("xyz", ligand_xyz, overwrite=True)
        
        ligands.append((smiles, ligand_xyz))
        print(f"Лиганд {i+1}: SMILES = {smiles}")
        print(f"         Файл = {ligand_xyz}")
        print()
    
    print(f"Всего найдено лигандов: {len(ligands)}")

# Запуск
xyz_filename = "/Users/egorilin/Desktop/Agents_2/app/ABAFOZ.xyz"
extract_ligands_split(xyz_filename)